<h3 style="text-align: center; background-color: #5AA647; color: white; padding: 8px; border-radius: 6px;">
NLP - Topic Modelling using LDA vectorizer
</h3

<div style="
    border:2px solid #4CAF50;
    padding:20px;
    border-radius:10px;
    width:550px;
    background-color:#f9f9f9;
    font-family:Arial;
">

<h2 style="margin-top:0;">📊 Topic Modeling — Simple Explanation</h2>

<p>
Topic modeling is a way for computers to automatically find hidden topics in a collection of text documents.
</p>

<p>
👉 <strong>In very simple terms:</strong><br>
It answers the question — 
<em>What are the main themes in this text data?</em>
</p>

<h3>✅ Example</h3>

<p>Imagine you have 1000 customer reviews.</p>

<ul>
    <li>🧾 Delivery Issues → late, delay, shipping</li>
    <li>📦 Product Quality → broken, good, durable</li>
    <li>💬 Customer Service → support, response, help</li>
</ul>

<p>
👉 Even though the reviews are mixed, the model <strong>discovers patterns automatically.</strong>
</p>

<h3>🧠 How it works (basic idea)</h3>
<ul>
    <li>Reads all text documents</li>
    <li>Looks for frequently occurring words together</li>
    <li>Groups related words into topics</li>
</ul>

<h3>⚙️ Common Algorithms</h3>
<ul>
    <li>LDA (Latent Dirichlet Allocation) → most common</li>
    <li>NMF (Non-negative Matrix Factorization) → easy and fast</li>
</ul>

<h3>✅ Output Example</h3>

<p><strong>Topic 1:</strong> [delivery, delay, shipping, late]</p>
<p><strong>Topic 2:</strong> [quality, product, good, bad]</p>

<p>
👉 You interpret these as:
</p>
<ul>
    <li><strong>Topic 1 → Delivery</strong></li>
    <li><strong>Topic 2 → Product Quality</strong></li>
</ul>

<h3>📌 Where it is used</h3>
<ul>
    <li>Customer feedback analysis</li>
    <li>Chat/message clustering</li>
    <li>News article grouping</li>
    <li>Recommendation systems</li>
</ul>

<h3>🧾 One-line Summary</h3>
<p>
👉 <strong>Topic modeling finds hidden themes in large text data automatically.</strong>
</p>

</div>

In [1]:
import pandas as pd
df = pd.read_csv("mabel.txt",header=None,encoding='utf8', on_bad_lines='skip')

In [2]:
# Remove the first row (often unwanted header or metadata row)
df = df.drop(0)

# Rename the columns to 'Date' and 'Chat'
df.columns = ['Date', 'Chat']

# Split the 'Chat' column into two parts using '-' as separator
# n=1 ensures only the first '-' is used for splitting
# expand=True creates separate columns
Message = df["Chat"].str.split("-", n=1, expand=True)

# Assign the first part of the split to a new column 'Time'
df["Time"] = Message[0]

# Now split the second part again using ':' to separate Name and message text
Message1 = Message[1].str.split(":", n=1, expand=True)

# Assign the first part as 'Name' (person who sent the message)
df["Name"] = Message1[0]

# Assign the second part as the actual chat message
df["Chat"] = Message1[1]

# Reorder/select only the required columns in the desired order
df = df[["Date", "Time", "Name", "Chat"]]

# Display the final cleaned DataFrame
df

,Date,Time,Name,Chat
1,05/12/19,1:42 pm,Mabel Infoziant,Hi this is Mabel we just spoke
2,05/12/19,1:42 pm,Mabel Infoziant,What’s your full name
3,05/12/19,1:42 pm,AR❤,Ramisha Rani K
4,05/12/19,1:42 pm,Mabel Infoziant,Ok
5,05/12/19,1:42 pm,Mabel Infoziant,ramisharanik@gmail.com
6,05/12/19,1:43 pm,Mabel Infoziant,Your email Id?
7,05/12/19,1:43 pm,AR❤,Yes Mam
8,05/12/19,1:43 pm,Mabel Infoziant,I will send 2 abstracts for u to start working
9,05/12/19,1:43 pm,AR❤,Yeah mam
10,05/12/19,1:43 pm,Mabel Infoziant,Give me the list that u have too


#### Counter VEctorizer Strarts here

In [3]:
# Import CountVectorizer (IMPORTANT for LDA)
from sklearn.feature_extraction.text import CountVectorizer

# Create count vectorizer
cv = CountVectorizer(
    max_df=0.95,       # ignore very frequent words
    min_df=2,          # ignore rare words
    stop_words='english'
)

# Create Document-Term Matrix (counts, not TF-IDF)
dtm = cv.fit_transform(df["Chat"])

#### LDA code changes

In [4]:
# Import LDA
from sklearn.decomposition import LatentDirichletAllocation

# Create LDA model
lda_model = LatentDirichletAllocation(
    n_components=5,    # number of topics
    random_state=42
)

# Fit model
lda_model.fit(dtm)

,n_components,5
,doc_topic_prior,None
,topic_word_prior,None
,learning_method,'batch'
,learning_decay,0.7
,learning_offset,10.0
,max_iter,10
,batch_size,128
,evaluate_every,-1
,total_samples,1000000.0
,perp_tol,0.1


#### Display Topics (Top Words)

In [5]:
# Print top words for each topic
for index, topic in enumerate(lda_model.components_):
    
    results = [cv.get_feature_names_out()[i] for i in topic.argsort()[-10:]]
    
    print(f"Topic {index}: {results}")

Topic 0: ['mam', 'number', 'need', 'ask', 'think', 'yes', 'ml', 'send', 'details', 'vignesh']
Topic 1: ['details', 'ask', 'share', 'phone', 'students', 'office', 'need', 'send', 'number', 'ok']
Topic 2: ['project', 'share', 'soon', 'finiliaze', 'yeah', 'kk', 'sure', 'read', 'abstract', 'mam']
Topic 3: ['finiliaze', 'ramisha', 'hi', 'mam', 'know', 'church', 'project', 'just', 'office', 'tomorrow']
Topic 4: ['mam', 'just', 'send', 'ramisha', 'church', 'soon', 'meeting', 'start', 'abstracts', 'hi']


#### Assign topics to dataframe

In [6]:
# Get topic distribution for each document
topic_results = lda_model.transform(dtm)

# Assign dominant topic
df["Topic"] = topic_results.argmax(axis=1)

#### Add Sentiment

In [7]:

from nltk.sentiment.vader import SentimentIntensityAnalyzer
sid = SentimentIntensityAnalyzer()

df["Sentiment"] = df["Chat"].apply(
    lambda x: sid.polarity_scores(x)['compound']
)

# Topic-wise sentiment
df.groupby("Topic")["Sentiment"].mean()
df

,Date,Time,Name,Chat,Topic,Sentiment
1,05/12/19,1:42 pm,Mabel Infoziant,Hi this is Mabel we just spoke,4,0.0000
2,05/12/19,1:42 pm,Mabel Infoziant,What’s your full name,0,0.0000
3,05/12/19,1:42 pm,AR❤,Ramisha Rani K,4,0.0000
4,05/12/19,1:42 pm,Mabel Infoziant,Ok,1,0.2960
5,05/12/19,1:42 pm,Mabel Infoziant,ramisharanik@gmail.com,0,0.0000
6,05/12/19,1:43 pm,Mabel Infoziant,Your email Id?,0,0.0000
7,05/12/19,1:43 pm,AR❤,Yes Mam,2,0.4019
8,05/12/19,1:43 pm,Mabel Infoziant,I will send 2 abstracts for u to start working,4,0.0000
9,05/12/19,1:43 pm,AR❤,Yeah mam,2,0.2960
10,05/12/19,1:43 pm,Mabel Infoziant,Give me the list that u have too,0,0.0000


In [14]:
for i, topic in enumerate(nmf_model.components_):
    words = [tfidf.get_feature_names_out()[j] for j in topic.argsort()[-10:]]
    print(f"Topic {i}: {words}")

Topic 0: ['share', 'soon', 'ask', 'yes', 'sure', 'read', 'abstract', 'kk', 'yeah', 'mam']
Topic 1: ['hi', 'kk', 'know', 'meeting', 'yes', 'just', 'send', 'tomorrow', 'mam', 'ok']
Topic 2: ['details', 'abstracts', 'start', 'students', 'phone', 'ask', 'office', 'number', 'vignesh', 'send']
Topic 3: ['ask', 'abstracts', 'number', 'mam', 'office', 'know', 'just', 'tomorrow', 'church', 'hi']
Topic 4: ['students', 'know', 'sure', 'share', 'soon', 'meeting', 'details', 'yes', 'need', 'ramisha']
